# OBS MNE Gradient Artifact Removal Demo

Este notebook carga un registro simultaneo EEG-fMRI (`fmrirestingec`), excluye el ultimo canal de ECG del pipeline, aplica la implementacion de PCA-OBS de MNE usando tiempos periodicos derivados del TR y muestra visualizaciones antes y despues.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import mne
import numpy as np

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from functions.obs_mne_ga import load_raw_eeglab, prepare_raw_for_obs, run_obs_pipeline_mne

In [2]:
DEFAULT_EEG_ROOT = Path(
    r"C:\Users\pedro\Documents\DOCTORADO_Pedro\Code\phD_Repository\data\raw\Dataset1\Simultaneous_EEG_fMRI\BIDS_dataset_EEG"
)
SUBJECT = "sub-007"
TASK = "fmrirestingec"
TR = 2.0
OFFSET_SAMPLES = 986
EXCLUDE_LAST_CHANNEL = True
N_COMPONENTS = 4
ANCHOR = "start"
N_JOBS = 1
DISPLAY_SECONDS = 20.0
PLOT_CHANNEL_INDEX = 9

In [3]:
def get_eeg_set_path(subject: str, task: str = TASK, eeg_root: Path = DEFAULT_EEG_ROOT) -> Path:
    eeg_path = eeg_root / subject / "eeg" / f"{subject}_task-{task}_eeg.set"
    if not eeg_path.exists():
        raise FileNotFoundError(f"EEG file not found: {eeg_path}")
    return eeg_path


def plot_channel_before_after(
    raw_before: mne.io.BaseRaw,
    raw_after: mne.io.BaseRaw,
    channel_index: int = 0,
    duration_s: float = DISPLAY_SECONDS,
) -> None:
    fs = float(raw_before.info["sfreq"])
    n_samples = min(int(round(duration_s * fs)), raw_before.n_times, raw_after.n_times)
    times = np.arange(n_samples) / fs

    before = raw_before.get_data(picks=[channel_index])[0, :n_samples]
    after = raw_after.get_data(picks=[channel_index])[0, :n_samples]
    channel_name = raw_before.ch_names[channel_index]

    fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
    axes[0].plot(times, before, linewidth=0.8)
    axes[0].set_title(f"Before MNE OBS | {channel_name}")
    axes[0].set_ylabel("Amplitude")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(times, after, linewidth=0.8, color="tab:orange")
    axes[1].set_title(f"After MNE OBS | {channel_name}")
    axes[1].set_xlabel("Time (s)")
    axes[1].set_ylabel("Amplitude")
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_overlay_before_after(
    raw_before: mne.io.BaseRaw,
    raw_after: mne.io.BaseRaw,
    channel_index: int = 0,
    duration_s: float = DISPLAY_SECONDS,
) -> None:
    fs = float(raw_before.info["sfreq"])
    n_samples = min(int(round(duration_s * fs)), raw_before.n_times, raw_after.n_times)
    times = np.arange(n_samples) / fs
    before = raw_before.get_data(picks=[channel_index])[0, :n_samples]
    after = raw_after.get_data(picks=[channel_index])[0, :n_samples]
    channel_name = raw_before.ch_names[channel_index]

    plt.figure(figsize=(14, 4))
    plt.plot(times, before, label="Before MNE OBS", linewidth=0.8, alpha=0.75)
    plt.plot(times, after, label="After MNE OBS", linewidth=0.8, alpha=0.75)
    plt.title(f"Overlay Before/After | {channel_name}")
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [4]:
eeg_path = get_eeg_set_path(SUBJECT)
raw = load_raw_eeglab(eeg_path)
raw_obs, picks = prepare_raw_for_obs(raw, exclude_last_channel=EXCLUDE_LAST_CHANNEL)

print(f"Subject: {SUBJECT}")
print(f"Task: {TASK}")
print(f"EEG path: {eeg_path}")
print(f"Original shape: {raw.get_data().shape}")
print(f"OBS input shape: {raw_obs.get_data().shape}")
print(f"Sampling frequency: {raw_obs.info['sfreq']} Hz")
print(f"Plot channel: {PLOT_CHANNEL_INDEX} ({raw_obs.ch_names[PLOT_CHANNEL_INDEX]})")
if EXCLUDE_LAST_CHANNEL:
    print(f"Excluded channel: {raw.ch_names[-1]}")

Subject: sub-007
Task: fmrirestingec
EEG path: C:\Users\pedro\Documents\DOCTORADO_Pedro\Code\phD_Repository\data\raw\Dataset1\Simultaneous_EEG_fMRI\BIDS_dataset_EEG\sub-007\eeg\sub-007_task-fmrirestingec_eeg.set
Original shape: (33, 628889)
OBS input shape: (32, 628889)
Sampling frequency: 1000.0 Hz
Plot channel: 9 (E10)
Excluded channel: ECG


## Visualizacion antes de limpiar

In [5]:
mne.viz.set_browser_backend("qt")
raw_obs.plot(
    n_channels=20,
    duration=10,
    show_scrollbars=True,
    block=True,
)

Using qt as 2D backend.
Channels marked as bad:
none


## Aplicacion del pipeline OBS de MNE

In [6]:
raw_clean, artifact_times = run_obs_pipeline_mne(
    raw_obs,
    TR=TR,
    offset_samples=OFFSET_SAMPLES,
    picks=picks,
    n_components=N_COMPONENTS,
    copy=True,
    n_jobs=N_JOBS,
    anchor=ANCHOR,
)

print(f"Artifact repetitions passed to MNE OBS: {len(artifact_times)}")
print(f"First 5 artifact times (s): {artifact_times[:5]}")

Artifact repetitions passed to MNE OBS: 314
First 5 artifact times (s): [0.986 2.986 4.986 6.986 8.986]


## Visualizacion despues de limpiar

In [7]:
raw_clean.plot(
    n_channels=20,
    duration=10,
    show_scrollbars=True,
    block=True,
)

Channels marked as bad:
none


In [ ]:
plot_channel_before_after(raw_obs, raw_clean, channel_index=PLOT_CHANNEL_INDEX, duration_s=DISPLAY_SECONDS)
plot_overlay_before_after(raw_obs, raw_clean, channel_index=PLOT_CHANNEL_INDEX, duration_s=DISPLAY_SECONDS)